
```
███╗   ███╗██╗███╗   ██╗███████╗ ██████╗ ██████╗ ██╗      █████╗ ██████╗
████╗ ████║██║████╗  ██║██╔════╝██╔════╝██╔═══██╗██║     ██╔══██╗██╔══██╗
██╔████╔██║██║██╔██╗ ██║█████╗  ██║     ██║   ██║██║     ███████║██████╔╝
██║╚██╔╝██║██║██║╚██╗██║██╔══╝  ██║     ██║   ██║██║     ██╔══██║██╔══██╗
██║ ╚═╝ ██║██║██║ ╚████║███████╗╚██████╗╚██████╔╝███████╗██║  ██║██████╔╝
╚═╝     ╚═╝╚═╝╚═╝  ╚═══╝╚══════╝ ╚═════╝ ╚═════╝ ╚══════╝╚═╝  ╚═╝╚═════╝
```
**Run a Minecraft Server on Google Colab!**

---

The script below will run your server. You'll have to create a server first to be able to use it - don't worry, the scripts below will do the majority of the work for you. You might also want to change the default region to your region, check below.

In [34]:
import os
import time
import queue
import socket
import hashlib
import shutil
import zipfile
import threading
import subprocess
import urllib.request
from pathlib import Path
from google.colab import drive

# ---------- Налаштування ----------
ARCHIVE_NAME = "ATM10SKY-2.2.0-server.zip"
SERVER_FOLDER = "ATM10SKY-2.2.0-clean"
MAX_RAM = "8G"
MIN_RAM = "2G"

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive")
archive = drive_root / ARCHIVE_NAME
server_dir = drive_root / SERVER_FOLDER

# Не запускаємо другу копію сервера.
with socket.socket() as connection:
    connection.settimeout(1)
    if connection.connect_ex(("127.0.0.1", 25565)) == 0:
        raise RuntimeError(
            "Порт 25565 зайнятий. Спочатку зупини попередній сервер."
        )

# ---------- Чисте розпакування ----------
marker = server_dir / ".atm10sky-extracted"

if not marker.exists():
    if not archive.is_file():
        raise FileNotFoundError(
            f"Завантаж {ARCHIVE_NAME} у корінь «Мій диск»."
        )

    if server_dir.exists() and any(server_dir.iterdir()):
        raise RuntimeError(
            f"Папка {server_dir} уже містить файли. "
            "Зміни SERVER_FOLDER на нову назву."
        )

    print("Розпаковую серверний архів…", flush=True)
    server_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive) as pack:
        # Перевірка шляхів усередині ZIP.
        root = server_dir.resolve()
        for entry in pack.infolist():
            target = (root / entry.filename).resolve()
            if not target.is_relative_to(root):
                raise RuntimeError("ZIP містить некоректний шлях.")

        if "startserver.sh" not in pack.namelist():
            raise RuntimeError(
                "В архіві немає startserver.sh у корені. "
                "Перевір, що це серверний ZIP ATM10SKY."
            )

        pack.extractall(server_dir)

    marker.write_text("ATM10SKY 2.2.0", encoding="utf-8")

if not (server_dir / "startserver.sh").is_file():
    raise RuntimeError("Не знайдено startserver.sh.")

print(f"Папка сервера: {server_dir}", flush=True)

# ---------- Умови Minecraft ----------
eula_file = server_dir / "eula.txt"
accepted = (
    eula_file.is_file()
    and any(
        line.strip().lower() == "eula=true"
        for line in eula_file.read_text(encoding="utf-8").splitlines()
    )
)

if not accepted:
    print("Minecraft EULA: https://www.minecraft.net/eula")
    answer = input(
        "Якщо прочитав і погоджуєшся, введи ТАК: "
    ).strip().upper()

    if answer != "ТАК":
        raise RuntimeError("Запуск скасовано: EULA не підтверджено.")

    eula_file.write_text("eula=true\n", encoding="utf-8")

# ---------- Видимий вивід установлення ----------
def run_visible(command):
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors="replace",
    )

    for line in process.stdout:
        print(line.rstrip(), flush=True)

    if process.wait() != 0:
        raise RuntimeError(
            f"Команда завершилася з помилкою: {command}"
        )

print("Установлюю Java 21…", flush=True)

run_visible(["sudo", "apt-get", "update", "-qq"])
run_visible([
    "sudo", "apt-get", "install", "-y", "-qq",
    "openjdk-21-jre-headless", "curl", "ca-certificates",
])

java_home = Path("/usr/lib/jvm/java-21-openjdk-amd64")
java_bin = java_home / "bin/java"

if not java_bin.is_file():
    raise RuntimeError("Java 21 не знайдено.")

run_visible([str(java_bin), "-version"])

# ---------- Пам'ять Minecraft ----------
jvm_file = server_dir / "user_jvm_args.txt"

if jvm_file.is_file():
    original = jvm_file.read_text(encoding="utf-8")
    backup = server_dir / "user_jvm_args.before-colab.txt"
    if not backup.exists():
        backup.write_text(original, encoding="utf-8")
else:
    original = ""

# Зберігаємо інші параметри, прибираємо попередні Xms/Xmx.
remaining = []
for line in original.splitlines():
    if line.lstrip().startswith("#"):
        remaining.append(line)
        continue

    tokens = [
        token for token in line.split()
        if not token.startswith(("-Xms", "-Xmx"))
    ]
    remaining.append(" ".join(tokens))

jvm_file.write_text(
    "\n".join(remaining)
    + f"\n-Xms{MIN_RAM}\n-Xmx{MAX_RAM}\n",
    encoding="utf-8",
)

# ---------- Playit без systemd ----------
tools_dir = Path("/content/atm-playit-tools")
tools_dir.mkdir(exist_ok=True)

def download_verified(filename, expected_sha256):
    target = tools_dir / filename

    def checksum():
        return hashlib.sha256(target.read_bytes()).hexdigest()

    if not target.exists() or checksum() != expected_sha256:
        print(f"Завантажую {filename}…", flush=True)
        urllib.request.urlretrieve(
            "https://github.com/playit-cloud/playit-agent/"
            f"releases/download/v1.0.10/{filename}",
            target,
        )

    if checksum() != expected_sha256:
        raise RuntimeError(f"Невірна контрольна сума: {filename}")

    target.chmod(0o755)
    return str(target)

playit_agent = download_verified(
    "playit-linux-amd64",
    "2df7d9f10227ab312b1ad341853db4e8a8243df5cfcdbae58713a4271711c339",
)

playit_cli = download_verified(
    "playit-cli-linux-amd64",
    "6fd54d147ae1d3232b22c1c1f4aa3d13cf16d889e840ca2d3f90b4f50a2e7301",
)

env = os.environ.copy()
env["JAVA_HOME"] = str(java_home)
env["PATH"] = str(java_home / "bin") + os.pathsep + env["PATH"]
env["ATM10_JAVA"] = str(java_bin)
env["ATM10_RESTART"] = "false"

# ---------- Вивід усіх процесів ----------
messages = queue.Queue()
readers = []

def read_output(process, label):
    try:
        for line in process.stdout:
            messages.put(f"[{label}] {line.rstrip()}")
    finally:
        process.stdout.close()

def launch(command, label):
    process = subprocess.Popen(
        command,
        cwd=str(server_dir),
        env=env,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors="replace",
        bufsize=1,
    )

    reader = threading.Thread(
        target=read_output,
        args=(process, label),
        daemon=True,
    )
    reader.start()
    readers.append(reader)
    return process

def show_messages(timeout=0.3):
    try:
        print(messages.get(timeout=timeout), flush=True)
    except queue.Empty:
        return

    for _ in range(200):
        try:
            print(messages.get_nowait(), flush=True)
        except queue.Empty:
            break

agent = None
setup = None
minecraft = None

try:
    print("Запускаю Playit…", flush=True)
    agent = launch([playit_agent], "PLAYIT")

    # Даємо агенту створити локальний IPC-сокет.
    ready_at = time.monotonic() + 3
    while time.monotonic() < ready_at:
        show_messages()

    if agent.poll() is not None:
        raise RuntimeError(
            "Агент Playit завершився. Перевір рядки [PLAYIT]."
        )

    secret_file = Path("/root/.config/playit_gg/playit.toml")
    if not secret_file.exists():
        setup = launch(
            [playit_cli, "--stdout", "setup"],
            "PLAYIT-SETUP",
        )
    else:
        print(
            "Знайдено попередню конфігурацію Playit.",
            flush=True,
        )

    print(
        "\nПерший запуск: відкрий посилання з [PLAYIT-SETUP].\n"
        "У панелі Playit налаштуй Minecraft Java:\n"
        "  Local address: 127.0.0.1\n"
        "  Local port: 25565\n"
        "  Proxy Protocol: вимкнено\n"
        "У гру вводь публічну адресу з панелі Playit.\n",
        flush=True,
    )

    print(
        "Запускаю ATM10SKY. Перший запуск також установлює NeoForge.",
        flush=True,
    )
    minecraft = launch(
        ["bash", "startserver.sh"],
        "MINECRAFT",
    )

    agent_warning_shown = False

    while minecraft.poll() is None:
        show_messages()

        if agent.poll() is not None and not agent_warning_shown:
            print(
                "УВАГА: Playit завершився. "
                "Зовнішнє підключення недоступне.",
                flush=True,
            )
            agent_warning_shown = True

    print(
        f"Minecraft завершився з кодом {minecraft.returncode}.",
        flush=True,
    )

except KeyboardInterrupt:
    print(
        "Отримано зупинку. Прошу Minecraft зберегти світ…",
        flush=True,
    )

finally:
    if minecraft is not None and minecraft.poll() is None:
        try:
            minecraft.stdin.write("stop\n")
            minecraft.stdin.flush()
        except (BrokenPipeError, OSError):
            pass

        deadline = time.monotonic() + 120
        while minecraft.poll() is None and time.monotonic() < deadline:
            show_messages()

        if minecraft.poll() is None:
            print(
                "Процес ще працює або встановлює NeoForge. "
                "Не запускай другу копію сервера.",
                flush=True,
            )

    for process in (setup, agent):
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()

    for reader in readers:
        reader.join(timeout=1)

    while not messages.empty():
        show_messages(timeout=0.01)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu

# Make-a-Server

The code below will download a server for you and accept the EULA. After running these scripts, your server will be ready to run.

**Download the Minecraft server**

The code below will download Paper, a high-performance fork of the Vanilla server.
Other server platforms can be used by placing the server.jar in the Drive folder manually.

In [ ]:
# Replace "1.19.2" with your desired server version.
# Available versions:
# - 1.19.2
# - 1.19.1
# - 1.19
# - 1.18.2
# - 1.18.1
# - 1.18
# - 1.17.1
# - 1.17
# - 1.16.4
# - 1.16.3
# - 1.15.2
# - 1.14.4
# - 1.13.2
# - 1.12.2
# - 1.11.2
# - 1.10.2
# - 1.9.4
# - 1.8.8
# Newer versions might work too, however this isn't guaranteed.
version = '1.19.2'
forge_version = "43.2.8"  # If you are going to use forge

#Chose server type. Currently available versions: fabric, paper, forge
server_type = 'paper'

from google.colab import drive
import requests
import json


drive.mount('/content/drive')

! mkdir "/content/drive/My Drive/Minecraft-server"
%cd "/content/drive/My Drive/Minecraft-server"

if server_type == "paper":
  a = requests.get("https://papermc.io/api/v2/projects/paper/versions/" + version)
  b = requests.get("https://papermc.io/api/v2/projects/paper/versions/" + version + "/builds/" + str(a.json()["builds"][-1]))
  print("https://papermc.io/api/v2/projects/paper/versions/" + version + "/builds/" + str(a.json()["builds"][-1]) + "/downloads/" + b.json()["downloads"]["application"]["name"])

  url = "https://papermc.io/api/v2/projects/paper/versions/" + version + "/builds/" + str(a.json()["builds"][-1]) + "/downloads/" + b.json()["downloads"]["application"]["name"]

  jar_name = "server.jar"
print('Downloading to Google Drive...')
elif server_type == "fabric":
  url = "https://maven.fabricmc.net/net/fabricmc/fabric-installer/0.7.4/fabric-installer-0.7.4.jar"
  jar_name = "fabric-installer.jar"
elif server_type == "forge":
  url = f"https://maven.minecraftforge.net/net/minecraftforge/forge/{version}-{forge_version}/forge-{version}-{forge_version}-installer.jar"
  jar_name = "server.jar"


r = requests.get(url)

if r.status_code is 200:
  with open('/content/drive/My Drive/Minecraft-server/' + jar_name, 'wb') as f:
    f.write(r.content)
else:
  print('Error '+ str(r.status_code) + '! Most likely you entered a unsupported version. Try running the code again if you think that shouldn\'t have happened.')

# Running specific install path.
if server_type == "fabric":
  !java -jar fabric-installer.jar server -mcversion $version -downloadMinecraft
elif server_type == "forge":
  !java -jar server.jar --installServer

colabconfig = {"server_type": server_type}
json.dump(colabconfig, open("colabconfig.json",'w'))

print('Done!')


**Automatically accept the EULA**

In [ ]:
# Please read the file stored in your server folder before running this command.
# Also, go to https://www.minecraft.net/en-us/eula to read Minecraft's EULA.

# Make sure Drive is mounted
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/My Drive/Minecraft-server"
!echo "eula=true" >> eula.txt

# Debug
SSH access to host OS - Thanks to [colab-ssh](https://github.com/WassimBenzarti/colab-ssh).

<p style="color:red;">YOU MIGHT GET BANNED</p>

In [ ]:
#@title Colab-ssh tunnel
#@markdown Execute this cell to open the ssh tunnel. Check [colab-ssh documentation](https://github.com/WassimBenzarti/colab-ssh) for more details.

# Install colab_ssh on google colab
!pip install colab_ssh --upgrade

from colab_ssh import launch_ssh_cloudflared, init_git_cloudflared
ssh_tunnel_password = "<PUT_YOUR_PASSWORD_HERE>" #@param {type: "string"}
launch_ssh_cloudflared(password=ssh_tunnel_password)

In [ ]:
#Get public address (ngrok)
! curl -s http://localhost:4040/api/tunnels | python3 -c \
    "import sys, json; print(json.load(sys.stdin)['tunnels'][0]['public_url'])"

In [ ]:
## For inspecting the minecraft server directory ##
%cd "/content/drive/My Drive/Minecraft-server"
!ls
